# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samhere3116-maker/ML-Pipeline/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row in the raw table (fact_content_daily_performance) = one content item, for one client, on one day. For my lane (Refresh/Content Opportunity Scoring), I aggregate these daily rows into one row per content item over a chosen window. I'm using March 2026 (month=2026-03) as my working month, since it's mid-panel — not the sealed final month.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Context (never a feature): content_hash_id, client_hash_id, report_date — used only to group and join.
Feature (knowable before the decision point): gsc_impressions, gsc_clicks, gsc_avg_position from the window before my target window; content_created_at (from dim_content).
Label/proxy: whether impressions in the last 30 days dropped more than 20% vs the prior 30 days — this and anything computed directly from it is never a feature.
Excluded: url_hash_id / keyword_hash_id (grouping only, never features, and never to be reverse-engineered); the _sample table (June 2026) — sealed as a future test month, not used for building labels now.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Setting up the connection to the warehouse

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, getpass
import duckdb
import numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY90 = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
MONTH_START, MONTH_END = '2026-03-01', '2026-03-31'


Query 1 — checking the grain

In [9]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {FACT}
    WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Duplicate grain rows found: {len(grain_check)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


Query 2 — row count and date span

In [10]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d,
           COUNT(DISTINCT content_hash_id) AS n_content, COUNT(DISTINCT client_hash_id) AS n_clients
    FROM {FACT}
    WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
""").df()
span

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_d,max_d,n_content,n_clients
0,9841378,2026-03-01,2026-03-31,331437,55


Query 3 — availability, IS TRUE:

In [11]:
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {FACT}
    WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
""").df()
avail['pct_available'] = avail['ga4_available_rows'] / avail['total_rows']
avail


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966.0,0.042064


Five features:
imp_prev30 — knowable because it only sums days strictly before the target window.
clk_prev30 — same, only prior-window days.
pos_prev30 — average position measured only in the prior window.
visible_queries — describes the page's historical query mix, not the outcome being predicted (caveat noted in Section 4).
content_age_days — content creation date is fixed and always in the past.

In [12]:
features = con.sql(f"""
    WITH windowed AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date > DATE '{MONTH_END}' - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN report_date <= DATE '{MONTH_END}' - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN report_date <= DATE '{MONTH_END}' - INTERVAL 30 DAY THEN gsc_clicks ELSE 0 END) AS clk_prev30,
            AVG(CASE WHEN report_date <= DATE '{MONTH_END}' - INTERVAL 30 DAY THEN gsc_avg_position END) AS pos_prev30
        FROM {FACT}
        WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
        GROUP BY 1, 2
        HAVING imp_prev30 >= 50
    )
    SELECT * FROM windowed
""").df()

qsignals = con.sql(f"""
    SELECT content_hash_id, ANY_VALUE(content_visible_query_count) AS visible_queries
    FROM {QUERY90} GROUP BY content_hash_id
""").df()

content_age = con.sql(f"""
    SELECT content_hash_id, DATE_DIFF('day', content_created_date, DATE '{MONTH_END}') AS content_age_days
    FROM {DIM_CONTENT}
""").df()

feat_frame = features.merge(qsignals, on='content_hash_id', how='left').merge(content_age, on='content_hash_id', how='left')
print(f"{len(feat_frame):,} content items in the feature frame")
feat_frame.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

29,925 content items in the feature frame


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_prev30,pos_prev30,visible_queries,content_age_days
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6398.0,125.0,1.0,4.928000,57.0,396
1,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5391.0,239.0,1.0,7.347280,43.0,396
2,client_73cda7b4e4f265ea,content_a7da352b73b02668,4753.0,191.0,0.0,7.832461,12.0,396
3,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,7583.0,126.0,2.0,6.166667,53.0,396
4,client_73cda7b4e4f265ea,content_20403327d8d9374c,3464.0,97.0,0.0,5.690722,38.0,396


The trap:

In [13]:
feat_frame['is_declining'] = (feat_frame['imp_last30'] < 0.8 * feat_frame['imp_prev30']).astype(int)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

cols = ['imp_prev30','clk_prev30','pos_prev30','visible_queries','content_age_days']
model_df = feat_frame.dropna(subset=cols + ['imp_last30'])
y = model_df['is_declining']

# Honest model
Xtr, Xte, ytr, yte = train_test_split(model_df[cols], y, test_size=0.25, random_state=42, stratify=y)
honest_auc = roc_auc_score(yte, LogisticRegression(max_iter=1000).fit(Xtr, ytr).predict_proba(Xte)[:,1])
print(f"Honest AUC: {honest_auc:.3f}")

# THE TRAP: sneak in imp_last30, the exact number the label is built from
leaky_cols = cols + ['imp_last30']
Xtr2, Xte2, ytr2, yte2 = train_test_split(model_df[leaky_cols], y, test_size=0.25, random_state=42, stratify=y)
leaky_auc = roc_auc_score(yte2, LogisticRegression(max_iter=1000).fit(Xtr2, ytr2).predict_proba(Xte2)[:,1])
print(f"Leaky AUC: {leaky_auc:.3f}")

Honest AUC: 0.977
Leaky AUC: 1.000


"Including imp_last30 as a feature pushed AUC from 0.911 to a perfect 1.000 — a dead giveaway of leakage, since imp_last30 is literally the number my label formula is built from. I removed it and kept 0.911 as the honest result. This confirms the five legitimate features do carry real predictive signal on their own, without needing to see the answer"

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice covers only March 2026 for whichever clients have data that month — it can't capture seasonality across the full 17-month history. visible_queries comes from a fixed trailing-90-day table whose window can overlap my 30-day target window, so I'm treating it as a soft signal, not a fully clean one, until it's rebuilt with proper window alignment.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.